# Student Assistant — Simple RAG Chatbot



## 1. Install libraries (run once)

For pytorch you dont have to run it if you already installed it during training.

In [ ]:
#%pip install -q "transformers>=4.46,<5" "peft>=0.13,<1" "accelerate>=1.1" sentencepiece safetensors faiss-cpu gradio numpy

#%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

## 2. Imports and context data

In [10]:
import json
import numpy as np
import torch
import faiss
import gradio as gr
from transformers import AutoTokenizer, AutoModel, T5ForConditionalGeneration
from peft import PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

with open("context_dataset/context_dataset.json", encoding="utf-8") as f:
    data = json.load(f)

contexts = [item["context"] for item in data]
print("Contexts:", len(contexts))

Device: cuda
Contexts: 370


## 3. BERT CLS embeddings and FAISS index


In [11]:
bert_tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/multi-qa-distilbert-dot-v1")
bert = AutoModel.from_pretrained("sentence-transformers/multi-qa-distilbert-dot-v1").to(device)
bert.eval()

def embed_texts(texts):
    vectors = []
    for i in range(0, len(texts), 16):
        batch = texts[i:i+16]
        inputs = bert_tokenizer(batch, return_tensors="pt", padding=True,
                                truncation=True, max_length=512).to(device)
        with torch.no_grad():
            outputs = bert(**inputs)
        vectors.extend(outputs.last_hidden_state[:, 0].cpu().numpy())
        
    return np.array(vectors, dtype="float32")

embeddings = embed_texts(contexts)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print("Indexed:", index.ntotal)

Indexed: 370


## 4. Retrieve the five most relevant contexts

In [12]:
def retrieve_contexts(question, k=5):
    query = embed_texts([question])
    scores, ids = index.search(query, k)
    return [contexts[i] for i in ids[0] if i != -1]

question = "How many semesters does the Electrical Engineering degree last?"

for i, passage in enumerate(retrieve_contexts(question), 1):
    print(f"{i}. {passage}\n")

1. The undergraduate program in Electrical and Electronic Engineering lasts ten semesters. The first six semesters focus on required foundation and specialization‑background courses, while the seventh to ninth semesters comprise specialization cycles. The tenth semester is devoted exclusively to the diploma thesis.

2. One semester corresponds to 30 ECTS credits. The tenth semester is dedicated exclusively to the diploma thesis. From the seventh semester, students select a specialization cycle: Energy; Communications and Networks; Electronics and Computer Systems.

3. One semester corresponds to 30 ECTS credits. From the seventh semester, students choose among three specialization cycles: Energy; Communications and Networks; Electronics and Computer Systems. Course categories include compulsory, compulsory elective, optional, and foreign language courses. The tenth semester is devoted exclusively to the diploma thesis, with procedures and evaluation criteria outlined.

4. One semester 

## 5. Load the recovered T5-LoRA model



In [13]:
tokenizer = AutoTokenizer.from_pretrained("t5-large")
base_model = T5ForConditionalGeneration.from_pretrained("t5-large")


Here i dont use the model directly from the file directory. I have uploaded it on a huggin face repository and call it from there.

In [14]:
tokenizer = AutoTokenizer.from_pretrained("t5-large")
base_model = T5ForConditionalGeneration.from_pretrained("t5-large")
#base_model.resize_token_embeddings(32100)
model = PeftModel.from_pretrained(base_model, "TechGenious/QA_T5uniwa")
model = model.to(device)
model.eval()
print("T5-LoRA loaded")

T5-LoRA loaded


## 6. Ask a question

In [15]:
def answer_question(question):
    passages = retrieve_contexts(question)
    context = " ".join(passages)
    prompt = f"question: {question} context: {context}"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=512).to(device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=64)
    return tokenizer.decode(output[0], skip_special_tokens=True)

print(answer_question("How many semesters does the Electrical Engineering degree last?"))

The undergraduate program in Electrical and Electronic Engineering lasts ten semesters.


## 7. Chat in Jupyter

The chat box and message bubbles appear directly underneath this cell. You can also use the local URL once you run the cell to see it on the browser.

In [68]:
def chat(message, history):
    return answer_question(message)

demo = gr.ChatInterface(fn=chat, title="AI Student Assistant")
demo.launch(inline=True)

d:\User\Διπλωματικη\hugging examples\.env\Lib\site-packages\gradio\chat_interface.py:345: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
